# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://docs.langchain.com/oss/python/integrations/document_loaders
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

## 주요 Document Loader

### Text file
- TextLoader 이용

In [3]:
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

# TextLoader를 이용해서 조회
## 객체 생성 - 읽을 파일의 경로를 지정.
loader = TextLoader(path, encoding="utf-8")

# 읽어오기
docs = loader.load()
# docs = loader.lazy_load()   # 읽은 문서를 사용할 때 읽는다.

print(type(docs), len(docs))
print(type(docs[0]))

# for doc in docs: # lazy_load() -> generator
#     print(type(doc))


<class 'list'> 1
<class 'langchain_core.documents.base.Document'>


In [ ]:
doc = docs[0]
print("문서정보조회: doc.metadata")
print(doc.metadata)
# 필요한 정보들을 추가할 수 있다. LLM에 전달할 프롬프트에 추가할 것들, 검색할 때 사용할 정보들.
# 메타데이터의 키 -> 사전에 설계가 필요
doc.metadata['category'] = "sports"
doc.metadata['tag'] = ["올림픽", "IOC", "동계올림픽", "하계올림픽"]
print(doc.metadata)

문서정보조회: doc.metadata
{'source': 'data/olympic.txt'}
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}


In [6]:
print("문서내용: doc.page_content")
print(doc.page_content[:200])

문서내용: doc.page_content
올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열


In [7]:
print("문서 식별자-id: doc.id")
print(doc.id)

문서 식별자-id: doc.id
None


### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [32]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

path = "data/novel/금_따는_콩밭_김유정.pdf"

# Document Loader 생성
## PyPDF
# loader = PyPDFLoader(path, mode="single")   # mode : single(전체를 하나의 문서로 읽기)
# loader = PyPDFLoader(path, mode="page")   # mode : page(page당 하나의 문서로 읽기)

## PyMuPDF
# loader = PyMuPDFLoader(path)

## PDFPlumber
loader = PDFPlumberLoader(path)

# 읽기
docs = loader.load()    # list[Dcument]
print(len(docs))
print(docs)

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

23
[Document(metadata={'source': 'data/novel/금_따는_콩밭_김유정.pdf', 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf', 'page': 0, 'total_pages': 23, 'Author': 'Unknown', 'CreationDate': "D:20241124070535+00'00'", 'Creator': 'Wikisource', 'ModDate': "D:20241124070537+00'00'", 'Producer': 'Wikisource', 'Title': '금 따는 콩밭'}, page_content='금 따는 콩밭\nExported from Wikisource on 2024년 11월 24일\n1\n'), Document(metadata={'source': 'data/novel/금_따는_콩밭_김유정.pdf', 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf', 'page': 1, 'total_pages': 23, 'Author': 'Unknown', 'CreationDate': "D:20241124070535+00'00'", 'Creator': 'Wikisource', 'ModDate': "D:20241124070537+00'00'", 'Producer': 'Wikisource', 'Title': '금 따는 콩밭'}, page_content='위키백과에 이 글\n🙝🙟\n과 관련된\n땅속 저 밑은 늘 음침하 자료가 있습니다.\n금 따는 콩밭\n다. 위키백과\n고달픈 간드렛불, 맥없이\n푸르끼하다.\n밤과 달라서 낮엔 되우 흐릿하였다.\n겉으로 황토 장벽으로 앞뒤좌우가 콕 막힌 좁직한 구뎅이.\n흡사히 무덤 속같이 귀중중하다. 싸늘한 침묵, 쿠더브레한\n흙내와 징그러운 냉기만이 그 속에 자욱하다.\n곡괭이는 뻔질 흙을 이르집는다. 암팡스러이 내려쪼며,\n퍽 퍽 퍼억.\n이렇게 메떨어진 소리뿐. 그러나 간간 우수수 하고 벽이 헐\n린다.\n영식이는 일손을 놓고 소맷자락을

In [30]:
from pprint import pprint
doc = docs[0]
# doc.metadata['author'] = '김유정'
pprint(doc.metadata)

{'author': 'Unknown',
 'creationDate': "D:20241124070535+00'00'",
 'creationdate': '2024-11-24T07:05:35+00:00',
 'creator': 'Wikisource',
 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20241124070537+00'00'",
 'moddate': '2024-11-24T07:05:37+00:00',
 'page': 0,
 'producer': 'Wikisource',
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'subject': '',
 'title': '금 따는 콩밭',
 'total_pages': 23,
 'trapped': ''}


In [27]:
print(doc.page_content)

1
금  따는  콩밭
Exported from Wikisource on 2024 년  11 월  24 일
2
위키백과
위키백과에  이  글
과  관련된 
자료가  있습니다 .
금  따는  콩밭
🙝🙟
땅속  저  밑은  늘  음침하
다 .
고달픈  간드렛불 , 맥없이
푸르끼하다 .
밤과  달라서  낮엔  되우  흐릿하였다 .
겉으로  황토  장벽으로  앞뒤좌우가  콕  막힌  좁직한  구뎅이 .
흡사히  무덤  속같이  귀중중하다 . 싸늘한  침묵 , 쿠더브레한
흙내와  징그러운  냉기만이  그  속에  자욱하다 .
곡괭이는  뻔질  흙을  이르집는다 . 암팡스러이  내려쪼며 ,
퍽  퍽  퍼억 .
이렇게  메떨어진  소리뿐 . 그러나  간간  우수수  하고  벽이  헐
린다 .
영식이는  일손을  놓고  소맷자락을  끌어당기어  얼굴의  땀을
훑는다 . 이놈의  줄이  언제나  잡힐는지  기가  찼다 . 흙  한줌을
집어  코밑에  바짝  들여대고  손가락으로  샅샅이  뒤져본다 . 완
연히  버력은  좀  변한  듯싶다 . 그러나  불통버력이  아주  다  풀
린  것도  아니었다 . 밀똥버력이라야  금이  온다는데  왜  이리
안  나오는지 .
곡괭이를  다시  집어든다 . 땅에  무릎을  꿇고  궁뎅이를  번쩍
든  채  식식거린다 . 곡괭이는  무작정  내려찍는다 . 바닥에서
3
물이  스미어  무르팍이  흔건히  젖었다 . 굿엎은  천판에서  흙방
울은  내리며  목덜미로  굴러든다 . 어떤  때에는  웃벽의  한쪽이
떨어지며  등을  탕  때리고  부서진다 .
그러나  그는  눈도  하나  깜짝하지  않는다 . 금을  캔다고  콩밭
하나를  다  잡쳤다 . 약이  올라서  죽을둥  살둥  눈이  뒤집힌  이
판이다 . 손바닥에  침을  탁  뱉고  곡괭이  자루를  한번  꼰아잡
더니  쉴  줄  모른다 .
등뒤에서는  흙  긁는  소리가  드윽드윽  난다 . 아직도  버력을
다  못  친  모양 . 이  자식이  일을  하나  시졸  하나 . 

### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [ ]:
from bs4 import BeautifulSoup

html_txt = """<html>
<body>
<p><b>제목</b> <span>내용</span></p>
<p>다음문단</p>
<div>다음 내용</div>
</body>
</html>
"""

soup = BeautifulSoup(html_txt)
txt1 = soup.get_text()   # 태그내의 text(content)만 추출한다.
txt2 = soup.get_text(strip=True)    # 태그사이의 공백(' ', 엔터, tab)을 제거.
txt3 = soup.get_text(strip=True, separator="\n")    # separator는 각 태그내의 text들을 지정한 구분자로 나눠서 반환.
print("---------기본---------")
print(txt1)
print("---------strip=True---------")
print(txt2)
print("---------strip=True, separator='\\n'---------")
print(txt3)

---------기본---------


제목 내용
다음문단
다음 내용



---------strip=True---------
제목내용다음문단다음 내용
---------strip=True, separator='\n'---------
제목
내용
다음문단
다음 내용


In [41]:
soup

<html>
<body>
<p><b>제목</b> <span>내용</span></p>
<p>다음문단</p>
<div>다음 내용</div>
</body>
</html>

In [49]:
from bs4 import SoupStrainer

strainer = SoupStrainer("span") # <span>
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<span>내용</span>

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import os

os.environ['USER_AGENT'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"

url = [
    "https://m.sports.naver.com/fifaworldcup2026/article/109/0005560415",
    "https://m.sports.naver.com/kfootball/article/660/0000111904"
]

loader = WebBaseLoader(
    # web_path=url[1],    # 1개 문서 조회
    web_path=url,   # 여러개 문서조회 list["url"들]
    # default_parser="lxml",  # BeautifulSoup(문서str, parser(html.parser))
    bs_kwargs={ # BeautifulSoup() 생성할 때 넣어줄 파라미터 설정.
        "parse_only":SoupStrainer(name="div",attrs={"class":"_article_content"})
    },
    bs_get_text_kwargs={    # get_text()의 파라미터 설정
        "strip":True,
        "separator":"\n\n"
    }
)

In [52]:
! uv pip install lxml
# lxml 설치후에 kernel 재시작

Checked 1 package in 11ms


In [62]:
docs = loader.load()

print(len(docs))
pprint(docs[0].metadata)
print(docs[0])

2
{'source': 'https://m.sports.naver.com/fifaworldcup2026/article/109/0005560415'}
page_content='[사진] 소셜 미디어

[OSEN=정승우 기자] 경기력 비판의 선을 넘어선 악성 댓글이 이어지자

설영우

측이 직접 입장을 냈다.

설영우 측은 25일 공식 소셜 미디어를 통해 "설영우 선수에게 항상 응원과 관심을 보내주시는 모든 분들께 감사드린다"라며 입장문을 공개했다.

이어 "경기력에 대한 의견과 평가는 스포츠의 일부이며, 건설적인 비판과 다양한 의견은 건강한 스포츠 문화의 중요한 요소"라고 밝혔다.

동시에 선을 넘은 비난에 대해서는 단호한 태도를 보였다. 설영우 측은 "최근 일부 댓글 및 메시지 중에는 욕설, 인신공격, 명예훼손, 허위사실 유포 등 건전한 의견 표현의 범위를 명백히 벗어난 사례들이 확인되고 있다"라고 전했다.

또 "이러한 행위는 어떠한 경우에도 정당화될 수 없으며, 선수 개인뿐 아니라 가족과 주변인들에게도 심각한 피해를 초래할 수 있다"라고 강조했다.

설영우를 향한 비판 여론은 멕시코전 이후 거세졌다.

홍명보

감독이 이끄는 한국 축구대표팀은 지난 19일(한국시간) 멕시코 과달라하라의 에스타디오 과달라하라에서 열린 2026 국제축구연맹(FIFA)

북중미 월드컵

조별리그 A조 2차전에서 멕시코에 0-1로 패했다.

한국은 경기 내내 치열하게 맞섰지만 후반 5분 실점 이후 끝내 동점골을 만들지 못했다. 설영우는 이날 왼쪽 윙백으로 선발 출전했다. 경기 후 일부 팬들은 설영우의 경기력을 지적했다. 문제는 비판이 곧바로 원색적인 비난과 인신공격으로 번졌다는 점이다.

경기력에 대한 평가는 가능하다. 월드컵 무대에서 대표팀 선수가 비판에서 자유로울 수는 없다. 설영우의 경기 내용에 아쉬움이 있었다고 보는 시선도 있을 수 있다. 다만 선수 개인의 인격을 공격하거나 부상을 기원하는 식의 댓글은 비판이 아니다.

설영우 측도 이 지점을 분명히 했다. 입장문에는 "건강하

#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
  ```bash
  시작 URL
  ├─ 내부 링크 1
  │   ├─ 내부 링크 1-1
  │   └─ 내부 링크 1-2
  ├─ 내부 링크 2
  └─ 내부 링크 3
  ```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
    ```

In [75]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

start_url = "http://docs.python.org/3/"

def extractor(html:str) -> str:
    """ 
    RecursiveUrlLoader는 HTML문서를 그대로 반환.
    HTML 문서를 str으로 받아서 원하는 부분만 추출하는 callback 함수
    Args:
        html(str):html(웹)문서
    Returns:
        str: html에서 body의 content(text)만 추출해서 반환
    """
    soup = BeautifulSoup(html)
    body_element = soup.select_one("div.body")  # python doc 사이트 - 내용이 <div class='body'> 내에 위치
    return body_element.get_text(strip=True, separator="\n") if body_element is not None else soup.get_text(strip=True, separator="\n")

loader = RecursiveUrlLoader(
    url = start_url,
    max_depth=2,    # 0: start_url, 1: start_url->link, 2: start_url->link->link
    prevent_outside=True,
    base_url=start_url,
    extractor=extractor
)

docs = loader.load()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_22040\1690908853.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html)
c:\SKN31\10_AI_Agent\.venv\Lib\site-packages\langchain_community\document_loaders\recursive_url_loader.py:44: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser t

In [76]:
len(docs)

28

In [78]:
idx = 1
doc = docs[idx]
doc.metadata

{'source': 'http://docs.python.org/3/using/index.html',
 'content_type': 'text/html',
 'title': 'Python Setup and Usage — Python 3.14.6 documentation',
 'description': 'This part of the documentation is devoted to general information on the setup of the Python environment on different platforms, the invocation of the interpreter and things that make working with P...',
 'language': 'en'}

In [79]:
print(doc.page_content)

Python Setup and Usage
¶
This part of the documentation is devoted to general information on the setup
of the Python environment on different platforms, the invocation of the
interpreter and things that make working with Python easier.
1. Command line and environment
1.1. Command line
1.2. Environment variables
2. Using Python on Unix platforms
2.1. Getting and installing the latest version of Python
2.2. Building Python
2.3. Python-related paths and files
2.4. Miscellaneous
2.5. Custom OpenSSL
3. Configure Python
3.1. Build Requirements
3.2. Generated files
3.3. Configure Options
3.4. Python Build System
3.5. Compiler and linker flags
4. Using Python on Windows
4.1. Python install manager
4.2. The embeddable package
4.3. The nuget.org packages
4.4. Alternative bundles
4.5. Supported Windows versions
4.6. Removing the MAX_PATH limitation
4.7. UTF-8 mode
4.8. Finding modules
4.9. Additional modules
4.10. Compiling Python on Windows
4.11. The full installer (deprecated)
4.12. Python laun

### <del>ArxivLoader</del>

- arxiv api가 업데이트 되고 ArxivLoader는 그에 맞춰 업데이트가 되지 않아 ArxivLoader는 정상적으로 실행되지 않는다. **arxiv api 를 이용해서 검색 후 pdf를 다운로드 받는다.**
- arxiv API: https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- 설치
  - `pip install arxiv`



In [ ]:
# arxiv lib를 이용해서 arxiv.org의 논물을 검색하여 다운로드 or 논물의 주요 정보를 조회한다.
# RAG 문서로 arxiv의 논물들이 필요할 경우 이용할 수 있다.

import arxiv
# 검색관련 설정
search = arxiv.Search(
    query="Advenced RAG",   # 검색어
    max_results=10, # 검색 논문 최대 개수
    sort_by=arxiv.SortCriterion.Relevance   # 정렬기준.
)
# 정렬기준
# arxiv.SortCriterion
## Relevance: query와 관련성이 높은(정확도)순서
## LastUpdateDate: 논문이 마지막으로 수정 업데이트된 날짜.
## SubmittedDate : 논문이 처음 제출된 날짜.

# 검색
client = arxiv.Client()
results = client.results(search)
print(results)  # Iterator

In [ ]:
paper = next(results)   # 첫번째(한개) 논문
print(type(paper))
# 아래 정보들 -> Document의 metadata
print(paper.title)  # 논문 제목
print(paper.authors) # 논문 저자들 list[Author]
print(paper.authors[0].name) # 첫번째 저자의 이름
print(paper.categories) # 논문의 분야(카테고리)
print(paper.summary) # 논문 요약 내용
print(paper.pdf_url) # 논문 PDF 파일의 url
print(paper.published) # 논문 발표 일시
print(paper.get_short_id()) # 논문 ID

<class 'arxiv.Result'>
RAG based Question-Answering for Contextual Response Prediction System
[arxiv.Result.Author('Sriram Veturi'), arxiv.Result.Author('Saurabh Vaichal'), arxiv.Result.Author('Reshma Lal Jagadheesh'), arxiv.Result.Author('Nafis Irtiza Tripto'), arxiv.Result.Author('Nian Yan')]
Sriram Veturi
['cs.CL', 'cs.IR']
Large Language Models (LLMs) have shown versatility in various Natural Language Processing (NLP) tasks, including their potential as effective question-answering systems. However, to provide precise and relevant information in response to specific customer queries in industry settings, LLMs require access to a comprehensive knowledge base to avoid hallucinations. Retrieval Augmented Generation (RAG) emerges as a promising technique to address this challenge. Yet, developing an accurate question-answering framework for real-world applications using RAG entails several challenges: 1) data availability issues, 2) evaluating the quality of generated content, and 3) t

In [87]:
# 논문 PDF 파일 다운로드
import os
import requests

save_dir = "data/papers"
os.makedirs(save_dir, exist_ok=True)

resp = requests.get(paper.pdf_url)
if resp.status_code == 200:
    with open(os.path.join(save_dir, paper.get_short_id()+".pdf"), "wb") as fo:
        fo.write(resp.content)

In [88]:
# 논문 PDF 파일 다운로드 함수
import os
import requests

def download_arxiv_paper(paper:arxiv.Result, dirpath:str):
    """검색결과(paper)를 다운로드 후 dirpath에 저장."""

    os.makedirs(dirpath, exist_ok=True)
    resp = requests.get(paper.pdf_url)
    if resp.status_code == 200:
        with open(os.path.join(dirpath, paper.get_short_id()+".pdf"), "wb") as fo:
            fo.write(resp.content)

In [89]:
save_dir = "data/papers"
for paper in results:
    download_arxiv_paper(paper, save_dir)

In [93]:
# arxiv 에서 검색어의 논문을 조회해서 다운 받은 후에 
# Document에 page_content에는 논문 내용을 metadata에는 위의 정보를 넣어서
# list[Document] 를 반환하는 함수.
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document


def load_arxiv_docs(
        query: str, # 검색어
        top_k: int=10, # 최대 검색 개수
        dirpath: str="."    # 논문저장할 디렉토리
) -> list[Document]:
    
    client = arxiv.Client()
    search = arxiv.Search(
        query=query, max_results=top_k, sort_by=arxiv.SortCriterion.Relevance
    )

    results = client.results(search)
    docs = [] # 생성한 Document(조회결과)를 담은 리스트
    for paper in results:
        # 다운로드
        download_arxiv_paper(paper, dirpath)
        # PDF 문서 load
        file_path = os.path.join(dirpath, paper.get_short_id()+".pdf")
        load = PyPDFLoader(file_path, mode="single")
        doc:Document = loader.load()[0] # list[Document]
        # 메타데이터는 paper의 정보로 변경.
        doc.metadata = {
            "title":paper.title,
            "authors":[a.name for a in paper.authors],
            "categories": paper.categories,
            "arxiv_url": paper.entry_id,    # 눈문 페이지 url
            "pdf_url": paper.pdf_url
        }

        docs.append(doc)
    return docs

In [94]:
docs = load_arxiv_docs(query="transformers", top_k=20, dirpath="data/papers/transformers")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_22040\1690908853.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html)


In [95]:
print(len(docs))

20


In [96]:
docs[0].metadata

{'title': 'Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants',
 'authors': ['Jose I. Aizpurua'],
 'categories': ['cs.LG'],
 'arxiv_url': 'http://arxiv.org/abs/2512.22190v1',
 'pdf_url': 'https://arxiv.org/pdf/2512.22190v1'}

### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [7]:
# Huggingface 로그인 - docling이 사용하는 모델은 로그인 후에 받을 수 있다.
from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv()

access_key = os.getenv("HUGGINGFACE_API_KEY")
login(access_key)

In [14]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# 현재 torch와 docling이 사용하는 OCR Lib (RapidOCR) 호환성 문제 때문에 ocr 기능을 끄는 설정을 한다.
pdf_option = PdfPipelineOptions()
pdf_option.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pdf_option
        )
    }
)

path = ["data/papers/2409.18313v5.pdf", "data/papers/2511.22858v1.pdf"]

loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN,
    converter=converter # ocr 기능을 껏기 때문에 직접 제작한 converter를 사용
)

In [15]:
docs = loader.load()

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [16]:
doc = docs[1]
doc.metadata

{'source': 'data/papers/2511.22858v1.pdf'}

In [17]:
print(doc.page_content)

## RAG System for Supporting Japanese Litigation Procedures: Faithful Response Generation Complying with Legal Norms

Yuya Ishihara 1 , * , Atsushi Keyaki 1 , * , Hiroaki Yamada 2 , Ryutaro Ohara 1,3 and Mihoko Sumida 1

1 Hitotsubashi University, Japan

2 Institute of Science Tokyo, Japan

3 Nakamura, Tsunoda &amp; Matsumoto, Japan

## Abstract

This study discusses the essential components that a Retrieval-Augmented Generation (RAG)-based LLM system should possess in order to support Japanese medical litigation procedures complying with legal norms. In litigation, expert commissioners, such as physicians, architects, accountants, and engineers, provide specialized knowledge to help judges clarify points of dispute. When considering the substitution of these expert roles with a RAG-based LLM system, the constraint of strict adherence to legal norms is imposed. Specifically, three requirements arise: (1) the retrieval module must retrieve appropriate external knowledge relevant to the 

In [18]:
from IPython.display import Markdown

Markdown(doc.page_content)

## RAG System for Supporting Japanese Litigation Procedures: Faithful Response Generation Complying with Legal Norms

Yuya Ishihara 1 , * , Atsushi Keyaki 1 , * , Hiroaki Yamada 2 , Ryutaro Ohara 1,3 and Mihoko Sumida 1

1 Hitotsubashi University, Japan

2 Institute of Science Tokyo, Japan

3 Nakamura, Tsunoda &amp; Matsumoto, Japan

## Abstract

This study discusses the essential components that a Retrieval-Augmented Generation (RAG)-based LLM system should possess in order to support Japanese medical litigation procedures complying with legal norms. In litigation, expert commissioners, such as physicians, architects, accountants, and engineers, provide specialized knowledge to help judges clarify points of dispute. When considering the substitution of these expert roles with a RAG-based LLM system, the constraint of strict adherence to legal norms is imposed. Specifically, three requirements arise: (1) the retrieval module must retrieve appropriate external knowledge relevant to the disputed issues in accordance with the principle prohibiting the use of private knowledge, (2) the responses generated must originate from the context provided by the RAG and remain faithful to that context, and (3) the retrieval module must reference external knowledge with appropriate timestamps corresponding to the issues at hand. This paper discusses the design of a RAG-based LLM system that satisfies these requirements.

## Keywords

Retrieval-Augmented Generation, Litigation Procedures, Legal Norms, Expert Knowledge, Information Retrieval

## 1. Introduction

In recent years, large language models (LLMs) have demonstrated remarkable advancements in their capabilities, leading to a growing movement toward their implementations into professional domains such as medicine and law. Since they are trained on extensive large text corpora, LLMs acquire a broad collection of knowledge throughout the training process [1, 2]. However, LLMs do not retain up-to-date information about events that occurred after their training period, and their knowledge of highly specialized or less common domains is not necessarily adequate. For these reasons, in professional domains such as legal[3] and medicine[4] , recent research has increasingly explored the use of RetrievalAugmented Generation (RAG) approaches, which exploits external knowledge to generate high-quality responses.

Our research group is studying a RAG-based LLM system to support medical litigation procedures in Japan. Normally, litigation process goes as follows, (i)arranging issues, (ii)fact finding, (iii)legal evaluation, (iv)writing reasons of outcome and (v)writing sentences1. In the process of (i)arrange issues, each claim submitted by the plaintiff and the defendant is examined to distinguish the points of agreement from those in dispute, thereby extracting points that should be the focus points of the litigation. In medical litigation, this process involves technical advisors who are medical professionals. These experts provide technical explanations regarding matters that may require witness examination or expert testimony during (ii)fact finding, attend sessions involving the discovery of evidence or witnesses, and offer explanations to judges to aid in assessing the reliability of evidence and witness testimonies presented by the parties. Particularly, because fact-finding by judges in medical litigation requires domain-specific medical expertise, identifying the matters that should be subject to expert testimony demands extensive referencing and analysis of a large volume of legal and medical claims and related documents. Consequently, the workload in this process is extremely high. According to Professor Murata Wataru (Chuo University), a former judge, Japanese courts prepare internal reference materials summarizing the case overview and the issues for expert testimony and opinion to facilitate judicial deliberation. Thus, support by computers, especially by the application of LLMs and RAG, is expected to significantly enhance the efficiency of these processes. Furthermore, in a RAG-based LLM system designed to support medical litigation procedures, it is essential not only to provide accurate information but also to ensure compliance with the judicial system's norms. Based on this premise, we propose a set of requirements that a legal RAG framework should satisfy in order to fulfill those normative principles.

This is a preprint version of a paper reviewed and accepted at BREV-RAG 2025: Beyond Relevance-based EValuation of RAG Systems, a SIGIR-AP 2025 workshop.

* Corresponding author.

* Corresponding author.

/envelope-open yuya.ishihara@r.hit-u.ac.jp (Y. Ishihara); a.keyaki@r.hit-u.ac.jp (A. Keyaki); yamada@comp.isct.ac.jp (H. Yamada); r.ohara@ntmlo.com (R. Ohara); m.sumida@r.hit-u.ac.jp (M. Sumida)

/orcid 0009-0003-8033-9927 (Y. Ishihara); 0000-0001-6495-117X (A. Keyaki); 0000-0002-1963-958X (H. Yamada); 0009-0000-8018-3895 (R. Ohara); 0000-0002-8531-2964 (M. Sumida)

Figure 1: process of civil litigation

## 2. Requirements in medical litigations

As a preliminary requirement, it is essential to retrieve external knowledge that is appropriate and relevant to the issues of dispute. Although the accuracy of the retrieval module is one of the established evaluation points in RAG systems, conducting appropriate expert testimony in the context of medical litigation further requires to comply with the norms of civil procedure and consideration of the domainspecific characteristics within the medical litigations.

## 2.1. Procedural Requirements for Use of Expert Knowledge

In Japan's civil litigation procedure, regardless of whether a case involves a specialized domain, the adversarial principle is adopted. Judges make decisions from a neutral point based solely on the claims and evidence submitted by both parties. Consequently, judges are restricted from relying on issues not submitted or raised by any of the parties or on knowledge not contained within the submitted evidence, as doing so would infringe upon the procedural rights of the parties. This adversarial principle, has a tension with the expectation of the litigation system that judges continually update their understanding of precedents, statutes, and social norms. The extent to which judges should be permitted to conduct judicial investigation and use privately acquired knowledge has thus become a subject of debate, particularly in the context of the ongoing digital transformation of society. This requirement applies not only to medical litigations but also to other types of litigation, particularly those that are highly specialized and involve expert advisory systems, such as intellectual property, construction and system development litigations.

Currently, it is recognized that when judges rely on specialized knowledge, domain-specific expert knowledge can be invoked without the presentation of evidence only if it has undergone critical verification by the relevant expert community, and the parties must be guranteed an opportunity to contest the use of such knowledge[5]. Furthermore, in the civil procedure systems of the United States, the United Kingdom, and Germany, the use of privately acquired expert knowledge is considered to be permitted under certain requirements, such as it is being commonly shared within the relevant expert community, or the implementation of procedural requirements including disclosure to both parties and the provision of an opportunity for comment[6]. Accordingly, in the context of RAG systems, the external knowledge sources should be limited to those that have been critically validated by expert communities, and access to such information must be controlled to ensure that it remains equally available to both parties involved in the proceedings.

## 2.2. Reliance of Expert Knowledge and Frequent Updates

Judges, by their professional responsibilities, are required to continually update their understanding of statutes, judicial precedents, and social norms. In the medical litigation, moreover, judges are additionally expected not only to address the specialized nature of the cases in charge but also to ensure the reliability of the data and evidence upon which their judgments rely. Furthermore, doctors who serve as technical advisors in medical domains provide technical explanations grounded in their medical expertise, and also update them. Due to continuous advances in medicine, even authoritative data sources such as well-established medical textbooks, peer-reviewed papers, and clinical guidelines issued by professional communities gradually lose their validity over time. According to Professor Shigeto Yonemura (The University of Tokyo), a leading authority in Japanese medical law and also a doctor, approximately twenty percent of such data becomes outdated within five years. While the knowledge acquired through pretraining inevitably becomes obsolete, continuously retraining LLMs to update domain-specific knowledge would be an inefficient approach. Therefore, it is essential that generated responses explicitly derive from and be faithful to the context provided through retrievals.

## 2.3. Issue-specific Reference Time

One of the reasons why expert testimony in arranging issues can become a complex procedure is that the applicable standard of expert knowledge differs depending on the issue in dispute. For example, when the issue concerns whether a physician was negligent or not, the judgment must be based on the medical knowledge, standard of care, and medical law valid at the time the incident happened. In contrast, when the issue concerns the causal relationship between a medical treatment and its outcome, the judgment should rely on the most up-to-date knowledge available at the close of oral proceedings[7]. Therefore, it is necessary to reference external knowledge corresponding to the appropriate time period relevant to the issue in focus.

Although authoritative data gradually lose their validity over time, the transition to new authorized knowledge does not occur abruptly at once. During the transitional period, multiple streams of expert knowledge that contradict each other may coexist until new data or precedents become widely acknowledged. Therefore, when retrieving appropriate sources in a RAG framework, it is necessary to Retrieval Module consider the expert knowledge that was valid at the relevant point in time.

Figure 2: Overview of the a RAG-based LLM System

Based on the above, this study addresses the realization of a 'norm-compliant RAG' system, focusing on: (1)controlling knowledge sources in compliance with procedural requirements concerning the use of expert knowledge, (2)attribution and faithfulness of generated responses to their information sources; and (3)appropriateness of the published time of referenced sources.

## 3. Related Work

## 3.1. Retrieval-Augmented Generation (RAG)

Since large language models (LLMs) are trained on extensive corpora, they acquire various forms of knowledge during the learning process [1, 2]. However, they do not retain up-to-date information, such as current events that occur after model training, and their coverage of specialized or less common knowledge is often insufficient. As a result, LLMs are known to generate responses containing misinformation, commonly referred to as hallucinations . Because the training corpora used in constructing LLMs may not adequately include domain-specific expertise, the presence of hallucinations is particularly likely when applying such models to specialized domains.

An overview of the RAG framework is presented in Figure 2. First, the user's input to the LLM is used as a query to retrieve relevant documents through a retrieval module. The retrieved documents are then provided to the LLM as contextual information, together with the user's input. The LLM generates a response while referring to these relevant documents. By leveraging high-quality external information through RAG, previous studies have reported improvements in task performance [13, 14] and reductions in hallucination occurrence [15, 16, 17, 3, 4].

Several approaches have been proposed to mitigate hallucinations, including improving the quality of training data [8], adjusting decoding strategies [9], enabling self-verification by the model [10, 11], and regenerating responses based on factual verification results [12]. Among these, Retrieval-Augmented Generation (RAG) [13, 14] has emerged as one of the most prominent and widely studied approaches.

However, completely suppressing hallucinations remains challenging even when using RAG. For example, a study on the application of RAG in the legal domain [3] reported that, although hallucinations can be mitigated through RAG, they cannot be entirely eliminated. Consequently, the study emphasizes the importance of expert responsibility in verifying the texts generated by LLMs when applying AI within the legal field.

In addition, [3] conducted an evaluation of hallucinations based on accuracy and factuality. Therefore, to the best of our knowledge, no prior research has focused on compliance with legal norms or on the appropriateness of the knowledge sources that substantiate such compliance, which constitutes the

## 3.2. Analysis of the Correspondence Between Information Sources and Responses

To verify whether the responses are faithfully generated based on the context provided by the RAG system, possible approaches include analysis using Data Attribution (DA) and evaluation methods related to response faithfulness.

Additionally, within the RAG framework, mechanisms have been proposed to evaluate the faithfulness of responses with respect to the provided context. For example, Ragas 1 enables the computation of a Faithfulness Score, which assesses the degree of consistency between the context and the generated response. The Faithfulness Score determines, through natural language inference, whether the content of the generated response is supported by the information contained in the given context. Specifically, the Faithfulness Score is calculated through the following procedure:

In existing studies on Data Attribution (DA) [18, 19], the focus of analysis has been on the pre-training data of LLMs, known as Training Data Attribution (TDA). In contrast, in the RAG-based LLM system examined in this study, knowledge derived from the RAG component and that originating from the LLM's pre-training data may exist in a competitive relationship, thereby requiring a more complex analytical approach.

1. Identify all the claims in the response.
3. Compute the faithfulness score using the formula:
2. Check each claim to see if it can be inferred from the retrieved context.

<!-- formula-not-decoded -->

## 4. Norm-compliant RAG

In this section, we discuss: (1) controlling knowledge sources in compliance with procedural requirements concerning the use of expert knowledge; (2) attribution and faithfulness of generated responses to their information sources; and (3) appropriateness of the published time of referenced sources, to realize a norm-compliant RAG.

## 4.1. Controlling Knowledge Sources in Compliance with Procedural Requirements

We restrict the use of external knowledge to sources that are acceptable according to civil litigation norms. We can achieve this control by controlling the scope of documents targeted by RAG retrieval and filtering the results. In assessing this aspect, we could simply label outputs derived from sources that deviate from the predefined scope as inappropriate.

## 4.2. Attribution and Faithfulness of Generated Responses to Information Sources

Expert knowledge is continuously updated over time. Thus, responses generated by relying solely on the model's knowledge acquired during its pre-training period can become easily outdated. RAG is the solution for this issue. To reinforce the effect of RAG, it is necessary to devise methods that generate responses faithful to the context (or documents) retrieved in the pipeline of the RAG approach. Possible approaches include explicit constraints through prompting and the introduction of chain-of-verification steps that check whether candidate sentences for generation are contained in the context.

To assess this aspect, we need to identify the data source or authority via DA analysis. We check whether the information contained in the outputs originates from RAG-derived knowledge or from pre-training. If it originates from pre-training, it is regarded as an inappropriate answer. Even if the generated output is based on RAG-derived knowledge, if it contradicts the knowledge in the source, it should be considered inappropriate. Thus, it is also necessary to assess the faithfulness of the outputs against the source.

[1 https://docs.ragas.io/en/stable/](https://docs.ragas.io/en/stable/)

## 4.3. Valid Time Period of Referenced Sources

The older documents can be outdated if they are overruled due to new discoveries and updates. A naive solution to this issue might be keeping the knowledge always updated to the latest version; however, this solution would not work in our legal RAG setting.

Thus, a legal RAG system should be able to recognize and manage the validity of time periods for sources correctly. Managing the time metadata of information sources is important, especially concerning when information is published and when it becomes invalid. We could achieve this by utilizing timestamps and citation networks.

The valid time periods of information sources vary depending on the types of issues raised in trials. For instance, if the issue of interest in a trial is a physician's negligence, the medical knowledge valid at the time of the physician's act can differ from that which is valid at the time of the trial, which is based on newer sources. If a system generates responses only according to the newest sources, it makes up an unrealistic conclusion based on knowledge unavailable at the time of the act in question. Moreover, the validity of time periods for sources matters not only in medical expert knowledge but also in legal expert knowledge, such as precedents and statutes.

When assessing this aspect, if a generated response is based on information with inappropriate timestamps that do not align with the input query, it is considered unsuitable.

## 5. Conclusion

We are developing a RAG-based LLM system to support medical litigation proceedings in Japan. Such a system must not just present accurate information, but also provide legally compliant responses to support expert testimony. To accommodate the requirements, we propose aspects of 'conformance to the norm' that a legal RAG system should satisfy. Specifically, we propose three aspects: (1) controlling knowledge sources in compliance with procedural requirements concerning the use of expert knowledge; (2) attribution and faithfulness of generated responses to their information sources; and (3) appropriateness of the published time of referenced sources.

Our future work includes proposing methods that satisfy each requirement, refining evaluation metrics, implementing them, and conducting experiments.

## Acknowledgments

We appreciate Prof. Shigeto Yonemura (The University of Tokyo), Prof. Wataru Murata (Chuo University), Prof. Shozo Ota (Meiji University), Prof. Simon Deakin (University of Cambridge), and Prof. Felix Steffek (University of Cambridge) for their helpful comments. This work was partially supported by Minji-Funsou-Shori-Kenkyukikin, the Japanese Society for the Promotion of Science Grantin-Aid for Scientific Research (B) (#23H03686, #25K03178) and Scientific Research (C) (#24K15066), and JST PRESTO (#JPMJPR236B).

## Declaration on Generative AI

During the preparation of this work, the authors used GPT-5 and Grammarly for grammar and style suggestions. After using this tool, the authors reviewed and edited the content and take full responsibility for the publication's content.

## References

- [1] F. Petroni, T. Rocktäschel, S. Riedel, P. Lewis, A. Bakhtin, Y. Wu, A. Miller, Language Models as Knowledge Bases?, in: Proc. of the EMNLP-IJCNLP 2019, 2019.
- [3] V. Magesh, F. Surani, M. Dahl, M. Suzgun, C. D. Manning, D. E. Ho, Hallucination-Free? Assessing the Reliability of Leading AI Legal Research Tools, Journal of Empirical Legal Studies 22 (2025) 216-242.
- [2] J. Wei, Y. Tay, R. Bommasani, C. Raffel, B. Zoph, S. Borgeaud, D. Yogatama, M. Bosma, D. Zhou, D. Metzler, E. H. Chi, T. Hashimoto, O. Vinyals, P. Liang, J. Dean, W. Fedus, Emergent abilities of large language models, Transactions on Machine Learning Research (TMLR) (2022) 2835-8856.
- [4] Y.-W. Chu, K. Zhang, C. Malon, M. R. Min, Reducing Hallucinations of Medical Multimodal Large Language Models with Visual Retrieval-Augmented Generation , in: Proc. of the AAAI 2025, 2025.
- [6] E. Sugiyama, Saibankan niyoru senmonchishiki no shushu to riyou (collection and use of expert knowledge by the judge, symposium: The discipline of civil judges in the exercise of their powers), Minso Zasshi (Journal of Civil Procedure) 69 (2023) 103 - 114.
- [5] G. Okanari, Saibankan no shichi riyou no kinshi (the prohibition of judge ' s use of private knowledge), Hougaku Zasshi :(Journal of Law) of Osaka City University 68 (2021) 1 - 66.
- [7] Y. Shirai, Mijukuji Moumakusyou to Ishi no Kashitu (Misdiagnosis of retinopathy of prematurity and Doctor ' s Negligence), Hanrei kara Manabu Minji-Jijitsu Nintei: Jurisuto Zoukan (Special Edition of Journal: Jurist: Learning from Case Law: Civil Fact-Finding) (2006) 252-256.
- [9] K. Li, O. Patel, F. Viégas, H. Pfister, M. Wattenberg, Inference-Time Intervention: Eliciting Truthful Answers from a Language Model, in: Proc. of the NeurIPS 2023, 2023.
- [8] A. Albalak, Y. Elazar, S. M. Xie, S. Longpre, N. Lambert, X. Wang, N. Muennighoff, B. Hou, L. Pan, H. Jeong, C. Raffel, S. Chang, T. Hashimoto, W. Y. Wang, A Survey on Data Selection for Language Models, arXiv:2402.16827, 2024.
- [10] P. Manakul, A. Liusie, M. Gales, SelfCheckGPT: Zero-Resource Black-Box Hallucination Detection for Generative Large Language Models, in: Proc. of the EMNLP 2023, 2023.
- [12] Y. Wang, R. G. Reddy, Z. M. Mujahid, A. Arora, A. Rubashevskii, J. Geng, O. M. Afzal, L. Pan, N. Borenstein, A. Pillai, I. Augenstein, I. Gurevych, P. Nakov, Factcheck-Bench: Fine-Grained Evaluation Benchmark for Automatic Fact-checkers, in: Proc. of the Findings of the EMNLP 2024, 2024.
- [11] X. Zhang, B. Peng, Y. Tian, J. Zhou, L. Jin, L. Song, H. Mi, H. Meng, Self-Alignment for Factuality: Mitigating Hallucinations in LLMs via Self-Evaluation, in: Proc. of the ACL 2024, 2024.
- [13] P. Lewis, E. Perez, A. Piktus, F. Petroni, V. Karpukhin, N. Goyal, H. Küttler, M. Lewis, W. tau Yih, T. Rocktäschel, S. Riedel, D. Kiela, Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, in: Proc. of the NIPS 2020, 2020.
- [15] S. T. I. Tonmoy, S. M. M. Zaman, V. Jain, A. Rani, V. Rawte, A. Chadha, A. Das, A Comprehensive Survey of Hallucination Mitigation Techniques in Large Language Models, arXiv:2401.01313, 2024.
- [14] Y. Gao, Y. Xiong, X. Gao, K. Jia, J. Pan, Y. Bi, Y. Dai, J. Sun, M. Wang, H. Wang, Retrieval-Augmented Generation for Large Language Models: A Survey, arXiv:2312.10997, 2023.
- [16] P. Lewis, E. Perez, A. Piktus, F. Petroni, V. Karpukhin, N. Goyal, H. Küttler, M. Lewis, W. tau Yih, T. Rocktäschel, S. Riedel, D. Kiela, Reducing Hallucination in Structured Outputs via RetrievalAugmented Generation, in: Proc. of the NAACL 2024, 2024.
- [18] G. Pruthi, F. Liu, S. Kale, M. Sundararajan, Estimating Training Data Influence by Tracing Gradient Descent, in: Proc. of the NeurIPS 2020, 2020.
- [17] W. Zhang, J. Zhang, Hallucination Mitigation for Retrieval-Augmented Large Language Models: A Review, Mathematics 13 (2025).
- [19] T. A. Chang, D. Rajagopal, T. Bolukbasi, L. Dixon, I. Tenney, Scalable Influence and Fact Tracing for Large Language Model Pretraining, in: Proc. of the ICLR 2025, 2025.

In [19]:
len(docs)

2

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [20]:
text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

splitter = CharacterTextSplitter(
    chunk_size = 60,
    chunk_overlap=10, # default : 200, chunk_overlab이 chunk_size보다 크면 안됨.
    # chunk_overlap은 separator 가 빈문자열일때 적용된다.
    separator=""    # 기본: '\n\n' => ""변경            "": chunk_size에 맞추겠다.
)

result = splitter.split_text(text)  # 입력 : str
print(type(result))
print(type(result[0]))
print(len(result))

<class 'list'>
<class 'str'>
4


In [27]:
for chunk in result:
    print(len(chunk), chunk)
    print("====================================================")

60 123456789012345678901234567890123456789012345678901234567890
60 1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM
60 DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg
55 이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ


In [29]:
doc = Document(page_content=text)
# 나눈 문서가 Document 객체일때
result_docs = splitter.split_documents([doc])

print(len(result_docs))
print(type(result_docs[0]))

4
<class 'langchain_core.documents.base.Document'>


In [31]:
for doc in result_docs:
    print(doc)
    print("================")

page_content='123456789012345678901234567890123456789012345678901234567890'
page_content='1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM'
page_content='DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg'
page_content='이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'


In [40]:
text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST.UVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 50,
    chunk_overlap = 10,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''] # 구분자 리스트를 직접 지정
    , is_separator_regex=True   # 구분자에 정규표현식 사용가능 여부.
)

result2 = splitter.split_text(text2)

In [42]:
print(splitter._separators)

['\n\n', '\n', '[\\.?!,~]', ' ', '']


In [43]:
for txt in result2:
    print(len(txt), txt)
    print("==========================================")

40 1234567890123456789012345678901234567890
29 12345678901234567890123456789
46 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST
7 .UVWXYZ
26 가나다라마바사아자차카타파하

아야어여오요우유으이
43 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ
9 RSTUVWXYZ
49 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
13 NOPQRSTUVWXYZ


In [49]:
## 문서 Load => Split
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

path = "data/olympic.txt"

# load
loader = TextLoader(path, encoding="UTF-8")
docs = loader.load()

print("Load한 문서 개수", len(docs))

# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''] # 구분자 리스트를 직접 지정
  , is_separator_regex=True   # 구분자에 정규표현식 사용가능 여부.
)
split_docs = splitter.split_documents(docs)
print("Split후 문서 개수", len(split_docs))

Load한 문서 개수 1
Split후 문서 개수 61


In [50]:
split_docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [51]:
# 문서 Load와 Split을 한번에 처리
docs = loader.load_and_split(splitter)

In [53]:
docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `o200k_base`: GPT-4 이후 모델들이 사용한 방식
    - `cl100k_base`: 초기 GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [64]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

loader = TextLoader(path, encoding="utf-8")
# from_tiktoken_encoder() - > OpenAI GPT모델을 사용할 경우 사용, GPT모델명 또는 Encoder(토크나이저)의 이름
# splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     # model_name="gpt-5",
#     encoding_name="o200k_base",
#     chunk_size=500, # 500 토큰 기준
#     chunk_overlap=50, # 50 토큰
# )

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # model_name="gpt-5",
    encoding_name="o200k_base",
    chunk_size=500, # 500 토큰 기준
    chunk_overlap=50, # 50 토큰
)

docs = loader.load_and_split(splitter)
print(len(docs))

Created a chunk of size 1027, which is longer than the specified 500
Created a chunk of size 981, which is longer than the specified 500
Created a chunk of size 881, which is longer than the specified 500
Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 703, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 925, which is longer than the specified 500
Created a chunk of size 661, which is longer than the specified 500
Created a chunk of size 1305, which is longer than the specified 500
Created a chunk of size 1052, which is longer than the specified 500


18


In [65]:
print(len(docs[0].page_content))

1726


In [67]:
# huggingface tokenizer 이용
from transformers import AutoTokenizer
model_id = "google/gemma-3-4b-it"
splitter_hf = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=AutoTokenizer.from_pretrained(model_id),    # transformers.Tokenizer 객체
    chunk_size=500,
    chunk_overlap=50
)

In [68]:
len(docs)

18

## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [69]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2
## 중주제2
- 기차
- 배
"""

In [79]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 나눌때 기준이 되는 Header를 설정.
# list(tuple(key,value)): key-Header 기호, value: 이름
header_to_split = [
    ("#","header1"),
    ("##","header2"),
    # ("###","header3"),
    # ("####","header4"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split,
    strip_headers=True, #default: True - 구분자 Header(제목)을 내용에 표시할 지 여부. True: 안넣는다.
)

docs = splitter.split_text(text)
len(docs)

3

In [80]:
docs

[Document(metadata={'header1': '대주제1'}, page_content='- 동물'),
 Document(metadata={'header1': '대주제1', 'header2': '중주제1'}, page_content='- 포유류  \n- 조류  \n### 소주제1\n- 개\n- 고양이\n- 까치\n- 독수리'),
 Document(metadata={'header1': '대주제2', 'header2': '중주제2'}, page_content='- 기차\n- 배')]

In [81]:
# olympic.md

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/olympic_wiki.md"
loader = TextLoader(path, encoding="utf-8")
header_to_split = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split
)

In [84]:
"\n".join(["a", "b", "c"])

'a\nb\nc'

In [89]:
docs = loader.load()

# list[Document] -> Document에서 page_content(읽은 text)를 조회해서 하나씩 str로 반환
doc_txt = "\n".join(doc.page_content for doc in docs)

# MarkdownHeaderSplitter는 split_document가 없고 split_text(str)만 제공
split_docs = splitter.split_text(doc_txt)
len(split_docs)

25

In [90]:
split_docs[0].metadata

{'h1': '올림픽'}

In [91]:
print(split_docs[0].page_content)

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  
또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제, 정치,